# 07 · Training Recipe (one strong model)

**Purpose.** Fine-tune one pretrained model with the v3 recipe - metadata fusion, layer-wise learning rates, mixup, EMA - and read its learning curves.

| | |
|---|---|
| **Reads** | `data/splits/*`, `data/cache/*` (optional) |
| **Writes** | `models/baseline/<model>_meta_seed42/`, `reports/results/baseline.json`, figures |
| **Run time (Colab T4)** | 25-45 min (ConvNeXt-S, 288 px) |
| **Next notebook** | 08 · Model Comparison |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

## Why v2 accuracy was low, and what v3 changes

| Problem seen in v2 | v3 fix | Where |
|---|---|---|
| small ImageNet-1k CNNs at 224 px | ConvNeXt-S / EfficientNetV2-S pretrained on **ImageNet-22k**, 288 px | `model.py` |
| one learning rate for the whole backbone, too low to adapt | **layer-wise LR decay**: last blocks learn fast, early blocks slowly | `train.py` |
| over-fitting on ~1,900 training images | stronger augmentation, **mixup**, drop-path, weight decay 0.05 | `data.py`, `train.py` |
| noisy last-epoch weights | **EMA** of weights (smoother, usually +1-2 points) | `train.py` |
| photo only | **age group, sex and body region** fused with the image features | `model.py` |
| camera colour differs per dataset | **Shades-of-Gray colour constancy** on every image | `data.py` |
| any photo forced into 6 classes | **"Other" class** + confidence threshold + skin check | notebooks 03, 10 |

## 1. Data and the training function

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dermalens.data import make_loader, resolve_images
from dermalens.utils import get_device, load_json, save_json

info = load_json(paths.splits / "class_info.json")       # classes + weights frozen in notebook 05
CLASSES = info["classes"]                                  # 6 targets + "Other"
WEIGHTS = [info["class_weights"][name] for name in CLASSES]
train = pd.read_csv(paths.splits / "train.csv")
val = pd.read_csv(paths.splits / "val.csv")
IMAGE_ROOT, PATH_COL = resolve_images(train, paths)        # cache if present, else raw files
DEVICE = get_device()
IMAGE_SIZE = CONFIG["data"]["image_size"]
BATCH, WORKERS = CONFIG["train"]["batch_size"], CONFIG["train"]["num_workers"]
USE_META, COLOR = CONFIG["data"]["use_metadata"], CONFIG["data"]["color_constancy"]


def loader(rows, train_mode=False, size=IMAGE_SIZE, use_meta=USE_META, classes=None, label_col="label",
           seed=0):
    """One DataLoader with the project settings (augmentation + metadata dropout only when training)."""
    return make_loader(rows, IMAGE_ROOT, classes or CLASSES, size, train=train_mode, batch_size=BATCH,
                       num_workers=WORKERS, path_col=PATH_COL, label_col=label_col, use_metadata=use_meta,
                       meta_dropout=CONFIG["data"]["meta_dropout"], color_constancy=COLOR, seed=seed)


print(f"train {len(train)} | val {len(val)} | classes {len(CLASSES)} | device {DEVICE} | "
      f"images from {IMAGE_ROOT} ({PATH_COL})")

In [ ]:
import time
from dermalens.evaluate import headline_metrics, predict_logits
from dermalens.model import build_model, count_parameters, zoo_entry
from dermalens.train import fit, model_from_checkpoint, train_config_from
from dermalens.utils import set_seed

TRAIN_CFG = train_config_from(CONFIG["train"])


def train_run(model_name, seed, use_meta, folder, rows=None, classes=None, weights=None, label_col="label"):
    """Train one model (or reuse it if done.json exists). Returns its validation record."""
    run_dir = paths.models / folder / f"{model_name}_{'meta' if use_meta else 'img'}_seed{seed}"
    if (run_dir / "done.json").exists():
        print("skip (done):", run_dir.name)
        return load_json(run_dir / "done.json")
    rows = rows if rows is not None else (train, val)
    classes, weights = classes or CLASSES, weights or WEIGHTS
    set_seed(seed)
    size = IMAGE_SIZE if zoo_entry(model_name)[2] == "cnn" else min(IMAGE_SIZE, 224)
    model = build_model(model_name, len(classes), CONFIG["model"]["pretrained"], use_meta,
                        CONFIG["model"]["drop_rate"], image_size=size)
    facts = {"model_name": model_name, "classes": classes, "image_size": size, "seed": seed,
             "use_metadata": use_meta}
    started = time.time()
    history = fit(model, loader(rows[0], True, size, use_meta, classes, label_col, seed),
                  loader(rows[1], False, size, use_meta, classes, label_col), weights, TRAIN_CFG, DEVICE,
                  run_dir, facts, seed=seed)
    history.to_csv(run_dir / "history.csv", index=False)
    best, _ = model_from_checkpoint(run_dir / "best.pt", DEVICE)
    logits, y_true = predict_logits(best, loader(rows[1], False, size, use_meta, classes, label_col), DEVICE)
    record = {"model": model_name, "seed": seed, "use_metadata": use_meta, "image_size": size,
              "checkpoint": str((run_dir / "best.pt").relative_to(paths.root)),
              "params_m": round(count_parameters(best)[0] / 1e6, 1),
              "minutes": round((time.time() - started) / 60, 1),
              **{k: round(float(v), 4) for k, v in headline_metrics(y_true, logits.argmax(1)).items()}}
    save_json(record, run_dir / "done.json")
    print(record)
    return record

## 2. Train

If Colab disconnects, run this cell again: training resumes from `last.pt`.

In [ ]:
MODEL_NAME = CONFIG["model"]["name"]
record = train_run(MODEL_NAME, CONFIG["seed"], USE_META, "baseline")
save_json(record, paths.results / "baseline.json")

## 3. Learning curves

*Healthy:* validation macro-F1 rises then flattens. *Over-fitting:* train accuracy near 100% while
validation falls - early stopping keeps the best epoch.

In [ ]:
from dermalens.evaluate import plot_training_curves

run_dir = (paths.root / record["checkpoint"]).parent
history = pd.read_csv(run_dir / "history.csv")
plot_training_curves(history, paths.figures / f"curves_baseline_{MODEL_NAME}.png", title=f"{MODEL_NAME}")
plt.show()

## 4. Validation confusion matrix
The test set stays untouched until notebook 09.

In [ ]:
from dermalens.evaluate import per_class_table, plot_confusion_matrix

model, facts = model_from_checkpoint(paths.root / record["checkpoint"], DEVICE)
logits, y_true = predict_logits(model, loader(val, size=facts["image_size"]), DEVICE)
display(per_class_table(y_true, logits.argmax(1), CLASSES))
plot_confusion_matrix(y_true, logits.argmax(1), CLASSES, paths.figures / "confusion_val_baseline.png",
                      title=f"{MODEL_NAME} - validation")
plt.show()